# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dishaprabhakar2006-blip/Flyrank/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [12]:
%pip -q install duckdb huggingface_hub


In [13]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [14]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [15]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("Rows where grain breaks (should be 0):", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain breaks (should be 0): 0


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [16]:
features_df = con.sql(f"""
    SELECT
      f.content_hash_id,
      f.client_hash_id,
      f.report_date,
      f.gsc_impressions,
      f.gsc_avg_position,
      f.gsc_clicks,
      date_diff('day', CAST(d.content_created_date AS DATE), CAST(f.report_date AS DATE)) AS content_age_days,
      d.word_count
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d
      ON f.content_hash_id = d.content_hash_id
    LIMIT 1000
""").df()
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_avg_position,gsc_clicks,content_age_days,word_count
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,2026-03-01,0,NaN,0,17,2999
1,content_67741cce996cfafa,client_62f4a7e64f5e0096,2026-03-01,0,NaN,0,17,3057
2,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,2026-03-01,4,3.500,0,17,2855
3,content_ac8663da7484669a,client_62f4a7e64f5e0096,2026-03-01,8,5.875,0,17,3281
4,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,2026-03-01,0,NaN,0,17,2779


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [17]:
q_span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
""").df()
q_span
q_avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
""").df()
q_avail

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966.0


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Build a proxy label from something NOT in our feature set: position tier at end of month
# (declining = average position worse than 20, i.e. fell off page 1-2)
features_df["is_declining_label"] = (features_df["gsc_avg_position"] > 20).astype(int)

# Honest features — observable BEFORE/independent of the label itself
honest_cols = ["gsc_impressions", "content_age_days", "word_count"]
X_honest = features_df[honest_cols].fillna(0)
y = features_df["is_declining_label"]
honest_model = DecisionTreeClassifier(max_depth=3).fit(X_honest, y)
print("HONEST score:", roc_auc_score(y, honest_model.predict_proba(X_honest)[:,1]))

# THE TRAP: deliberately add gsc_avg_position itself — the exact column the label is derived from
leaky_cols = honest_cols + ["gsc_avg_position"]
X_leaky = features_df[leaky_cols].fillna(0)
leaky_model = DecisionTreeClassifier(max_depth=3).fit(X_leaky, y)
print("LEAKY score (gsc_avg_position included):", roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:,1]))

HONEST score: 0.9056980851459379
LEAKY score (gsc_avg_position included): 1.0


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
